# arcaid3-mount-diag v3: read-throughput canary

v1/v2 only checked that `/kaggle/input` is mounted. Discussion 751598 reports
the mounts appear instantly but model weights take ~40 min to read (was 4-5 min),
so vLLM is not up in time and runs score 0.00. This version times a full,
uncached read of every attached model/dataset file and prints a verdict.
Results are also written to `/kaggle/working/read_bench.json`.

In [ ]:
import os, time, subprocess
t0 = time.time()
P = "/kaggle/input/datasets/dfranzen/taaf-kaggle-source-bundle-copy"
for i in range(61):
    ok = os.path.isdir(P)
    print(f"t={time.time()-t0:5.0f}s exists={ok} input={sorted(os.listdir('/kaggle/input')) if os.path.isdir('/kaggle/input') else None}", flush=True)
    if ok:
        print(sorted(os.listdir(P)))
        break
    time.sleep(10)
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth <= 2: print(root, sorted(dirs)[:10], len(files))
    if depth >= 2: dirs[:] = []
print(subprocess.run(["bash","-lc","mount | grep kaggle/input; nvidia-smi -L; df -h /kaggle/working /tmp"], capture_output=True, text=True).stdout)

In [ ]:
import os, time, glob, json
# Roots read in vLLM startup order (main model first). Stop after BUDGET_S so a
# degraded mount cannot burn the whole session; the verdict extrapolates.
ROOTS = [
    "/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround",
    "/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter",
    "/kaggle/input/datasets/dfranzen/pennyroyal-v253",
]
BUDGET_S = 45 * 60
CHUNK = 64 << 20
GiB, MiB = 1 << 30, 1 << 20

results, t_all = [], time.time()
for r in ROOTS:
    files = sorted(f for f in glob.glob(f"{r}/**/*", recursive=True) if os.path.isfile(f))
    tot = sum(os.path.getsize(f) for f in files)
    print(f"\n== {os.path.basename(r)}: {len(files)} files, {tot/GiB:.2f} GiB", flush=True)
    t, n, next_report, ttfb, stopped = time.time(), 0, GiB, None, False
    for f in files:
        with open(f, "rb", buffering=0) as fh:
            while (b := fh.read(CHUNK)):
                if ttfb is None: ttfb = time.time() - t
                n += len(b)
                if n >= next_report:
                    el = time.time() - t
                    print(f"  {n/GiB:6.1f}/{tot/GiB:.1f} GiB  {n/MiB/el:6.0f} MiB/s  t={el:5.0f}s", flush=True)
                    next_report += GiB
                if time.time() - t_all > BUDGET_S:
                    stopped = True; break
        if stopped: break
    el = max(time.time() - t, 1e-6)
    rate = n / MiB / el
    if not files:
        print(f"!! no files under {r} (not attached or not mounted)", flush=True)
    est_full = tot / MiB / rate if rate else (0.0 if tot == 0 else float("inf"))
    res = dict(root=r, files=len(files), gib=round(tot/GiB, 2), read_gib=round(n/GiB, 2),
               secs=round(el, 1), mib_s=round(rate, 1), ttfb_s=round(ttfb or 0, 2),
               est_full_load_min=round(est_full/60, 1), stopped_by_budget=stopped)
    results.append(res)
    print(f"DONE {res}", flush=True)
    if stopped:
        print(f"!! budget of {BUDGET_S//60} min hit; remaining roots skipped", flush=True)
        break

est_total = sum(x["est_full_load_min"] for x in results)
verdict = "OK" if est_total <= 10 else ("SLOW" if est_total <= 25 else "DEGRADED")
summary = dict(verdict=verdict, est_total_load_min=round(est_total, 1),
               wall_min=round((time.time()-t_all)/60, 1), results=results,
               utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
os.makedirs("/kaggle/working", exist_ok=True)
json.dump(summary, open("/kaggle/working/read_bench.json", "w"), indent=2)
print("\n" + "=" * 60)
print(f"VERDICT: {verdict}  estimated weight load ~{est_total:.1f} min "
      "(healthy was 4-5 min; >25 min means agents will likely time out waiting for vLLM)")
print("=" * 60)